# Module 3 — Topic 5: Choosing the Right Chart
## Live Demo Notebook

**Scenario:** You have the Jumia Nigeria sales dataset (500 orders).
For each of the five questions below, we'll show the WRONG chart first — then build the RIGHT chart.

---

**Five questions to answer:**
1. What is the distribution of order revenues?
2. Which product category generates the most total revenue?
3. How many orders have each delivery status?
4. Is there a relationship between unit price and quantity ordered?
5. What is the payment method mix across the top 5 states?

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('jumia_sales.csv')
print('Shape:', df.shape)
df.head(3)

---
## Q1: What is the distribution of order revenues?
### WRONG → RIGHT: Bar chart vs Histogram

In [ ]:
# WRONG — a bar chart of individual revenue values shows nothing useful
df_sample = df.sort_values('revenue').head(50).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(df_sample)), df_sample['revenue'], color='steelblue')
ax.set_title('❌ WRONG: Revenue Values as Bar Chart')
ax.set_xlabel('Order index')
ax.set_ylabel('Revenue (₦)')
plt.tight_layout()
plt.show()

print('What question does this chart answer? Nothing useful.')
print('You cannot see the shape, the typical value, or the spread.')

In [ ]:
# RIGHT — histogram with KDE shows distribution shape immediately
fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(data=df, x='revenue', bins=30, kde=True, color='steelblue', ax=ax)
ax.axvline(df['revenue'].median(), color='orange', linestyle='--', linewidth=2,
           label=f"Median: ₦{df['revenue'].median():,.0f}")

ax.set_title('✅ RIGHT: Distribution of Jumia Order Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue per Order (₦)', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.legend()

plt.tight_layout()
plt.show()

print(f"Skewness: {df['revenue'].skew():.2f} — right-skewed")
print("The histogram shows: most orders are low-value, a small number are very high.")

---
## Q2: Which product category generates the most total revenue?
### WRONG → RIGHT: Pie chart vs Bar chart

In [ ]:
# WRONG — pie chart with 8 categories is almost unreadable
category_rev = df.groupby('category')['revenue'].sum()

fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(category_rev.values, labels=category_rev.index, autopct='%1.0f%%',
       startangle=90)
ax.set_title('❌ WRONG: Revenue by Category — Pie Chart')
plt.tight_layout()
plt.show()

print("Can you tell which category is #1 and which is #8 just by looking?")
print("Most people can't — the slices are too similar in size.")

In [ ]:
# RIGHT — horizontal bar chart: ranking is immediately obvious
category_rev_sorted = df.groupby('category')['revenue'].sum().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(category_rev_sorted.index, category_rev_sorted.values,
               color='coral', edgecolor='white')

# Add value labels
for bar in bars:
    width = bar.get_width()
    ax.text(width + 50000, bar.get_y() + bar.get_height() / 2,
            f'₦{width/1e6:.1f}M', va='center', fontsize=9)

ax.set_title('✅ RIGHT: Total Revenue by Product Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (₦)', fontsize=12)
ax.set_ylabel('Category', fontsize=12)

plt.tight_layout()
plt.show()

print('Ranking is immediately obvious. The bar chart answers the question directly.')

---
## Q3: How many orders have each delivery status?
### WRONG → RIGHT: Scatter plot vs Bar chart

In [ ]:
# WRONG — scatter plot with a categorical x-axis makes no sense
status_counts = df['delivery_status'].value_counts().reset_index()
status_counts.columns = ['delivery_status', 'count']

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(status_counts['delivery_status'], status_counts['count'],
           s=200, color='mediumpurple')
ax.set_title('❌ WRONG: Delivery Status Counts — Scatter Plot')
ax.set_xlabel('Delivery Status')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print("Scatter plots require numeric x-axes.")
print("'Delivered', 'Shipped', etc. have no numerical position — this layout is misleading.")

In [ ]:
# RIGHT — bar chart for categorical counts
status_order = df['delivery_status'].value_counts().index

fig, ax = plt.subplots(figsize=(9, 4))
sns.countplot(data=df, x='delivery_status', order=status_order, palette='muted', ax=ax)

ax.set_title('✅ RIGHT: Number of Orders by Delivery Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Delivery Status', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---
## Q4: Is there a relationship between unit price and quantity ordered?
### WRONG → RIGHT: Line chart vs Scatter plot

In [ ]:
# WRONG — line chart connecting 500 dots in price order is meaningless
df_sorted = df.sort_values('unit_price').head(100)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sorted['unit_price'], df_sorted['quantity'],
        linewidth=0.7, color='steelblue', marker='o', markersize=3)
ax.set_title('❌ WRONG: Unit Price vs Quantity — Line Chart')
ax.set_xlabel('Unit Price (₦)')
ax.set_ylabel('Quantity')
plt.tight_layout()
plt.show()

print("This zigzag conveys no insight. The line implies a continuous trend — but there is none here.")

In [ ]:
# RIGHT — scatter plot reveals whether a pattern exists
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(df['unit_price'], df['quantity'], alpha=0.35, s=50, color='mediumpurple', edgecolors='none')

ax.set_title('✅ RIGHT: Unit Price vs Quantity Ordered', fontsize=14, fontweight='bold')
ax.set_xlabel('Unit Price (₦)', fontsize=12)
ax.set_ylabel('Quantity Ordered', fontsize=12)

plt.tight_layout()
plt.show()

corr = df['unit_price'].corr(df['quantity'])
print(f'Correlation (unit_price vs quantity): {corr:.3f}')
print('The scatter plot shows whether a visual pattern exists before we compute the correlation.')

---
## Q5: What is the payment method mix across the top 5 states?
### WRONG → RIGHT: Multiple pie charts vs Stacked bar chart

In [ ]:
# WRONG — five separate pie charts, impossible to compare across states
top5_states = df['state'].value_counts().head(5).index.tolist()

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, state in zip(axes, top5_states):
    state_data = df[df['state'] == state]['payment_method'].value_counts()
    ax.pie(state_data.values, labels=state_data.index, autopct='%1.0f%%',
           textprops={'fontsize': 7})
    ax.set_title(state, fontsize=10)

fig.suptitle('❌ WRONG: Payment Mix by State — Five Pie Charts', fontsize=13)
plt.tight_layout()
plt.show()

print("Can you compare payment patterns across states? Almost impossible.")

In [ ]:
# RIGHT — stacked bar chart: cross-state comparison is immediate
top5_df = df[df['state'].isin(top5_states)]
stacked  = top5_df.groupby(['state', 'payment_method']).size().unstack(fill_value=0)
# Normalise to proportions
stacked_pct = stacked.div(stacked.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5))
stacked_pct.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='white')

ax.set_title('✅ RIGHT: Payment Method Mix by State (% of Orders)', fontsize=14, fontweight='bold')
ax.set_xlabel('State', fontsize=12)
ax.set_ylabel('% of Orders', fontsize=12)
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Payment Method', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

print("Now you can compare payment patterns across all 5 states simultaneously.")

---
## Bonus: The Truncated Y-Axis Trap

In [ ]:
# Same data — completely different impression depending on y-axis
top3 = df.groupby('category')['revenue'].sum().nlargest(3)
cats = top3.index.tolist()
vals = top3.values.tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: truncated y-axis (misleading)
ax1.bar(cats, vals, color='steelblue', edgecolor='white')
ax1.set_ylim(min(vals) * 0.97, max(vals) * 1.01)  # small window
ax1.set_title('❌ MISLEADING: Truncated Y-Axis', fontsize=12, fontweight='bold', color='red')
ax1.set_ylabel('Total Revenue (₦)')
ax1.tick_params(axis='x', rotation=15)

# Right: full y-axis (honest)
ax2.bar(cats, vals, color='steelblue', edgecolor='white')
ax2.set_ylim(0, max(vals) * 1.15)
ax2.set_title('✅ HONEST: Y-Axis Starts at Zero', fontsize=12, fontweight='bold', color='green')
ax2.set_ylabel('Total Revenue (₦)')
ax2.tick_params(axis='x', rotation=15)

fig.suptitle('Same data — completely different story', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Min value: ₦{min(vals):,.0f}')
print(f'Max value: ₦{max(vals):,.0f}')
print(f'Difference: ₦{max(vals)-min(vals):,.0f} ({(max(vals)/min(vals)-1)*100:.1f}% more)')
print()
print("The left chart makes this look like a massive gap.")
print("The right chart shows the actual scale of the difference.")

---
## Summary

**Five wrong → right corrections:**

| Question | Wrong chart | Right chart | Why |
|---|---|---|---|
| Distribution | Bar chart of values | Histogram | Shows shape, not individual bars |
| Category ranking | Pie chart (8 slices) | Horizontal bar | Ranking immediately visible |
| Category counts | Scatter plot | Bar / countplot | Scatter needs numeric x-axis |
| Numeric relationship | Line chart | Scatter plot | Line implies sequence/time |
| Multi-group composition | Multiple pie charts | Stacked bar | Enables cross-group comparison |

**Bonus:** Always start the y-axis at zero for bar charts. A truncated axis distorts the visual impression.

**Next — Topic 6:** Data Storytelling. We take ONE key finding from this dataset and build a professional, annotated explanatory chart around it.